<a href="https://colab.research.google.com/github/shibataryoma22-hub/-udemy/blob/main/deeplearning_easy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#データの読み込み

In [6]:
import numpy as np
from sklearn import datasets
from sklearn.datasets import load_iris

In [9]:
#irisデータセットの読み込み
iris = load_iris()
print(iris.data[:10])
print(iris.target[:10])
print(iris.data.shape)



[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]
 [5.4 3.9 1.7 0.4]
 [4.6 3.4 1.4 0.3]
 [5.  3.4 1.5 0.2]
 [4.4 2.9 1.4 0.2]
 [4.9 3.1 1.5 0.1]]
[0 0 0 0 0 0 0 0 0 0]
(150, 4)


In [21]:
#データの前処理
from sklearn import preprocessing
import torch
import torch.nn.functional as F

In [17]:
scaler = preprocessing.StandardScaler()
scaler.fit(iris.data)
x = scaler.transform(iris.data) #irisデータを標準化
print(x[:10])

[[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]
 [-1.50652052  0.09821729 -1.2833891  -1.3154443 ]
 [-1.02184904  1.24920112 -1.34022653 -1.3154443 ]
 [-0.53717756  1.93979142 -1.16971425 -1.05217993]
 [-1.50652052  0.78880759 -1.34022653 -1.18381211]
 [-1.02184904  0.78880759 -1.2833891  -1.3154443 ]
 [-1.74885626 -0.36217625 -1.34022653 -1.3154443 ]
 [-1.14301691  0.09821729 -1.2833891  -1.44707648]]


In [28]:
#Tensorに変換（Tensorじゃないとpytorchでは動作しない）
#TensorはPytorch版のnumpy配列と考えればよい
x = torch.tensor(x, dtype = torch.float32)
#ラベル longで64bit整数に変換するとラベル使用で都合がよい
t = torch.tensor(iris.target, dtype = torch.long) #64bit整数
#onehot表現(irisのクラス数がnum_classes = 3)
t_onehot = F.one_hot(t, num_classes= 3)
print(t_onehot[:10])

tensor([[1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]])


訓練データとテストデータに分割する

In [29]:
from sklearn.model_selection import train_test_split

In [33]:
x_train, x_test, t_train, t_test = train_test_split(x, t, train_size = 0.75)
print(x_train.shape)
print(x_test.shape)
print(t_train.shape)
print(t_test.shape)

torch.Size([112, 4])
torch.Size([38, 4])
torch.Size([112])
torch.Size([38])


## モデルの構築


In [ ]:
'''
from keras.models import Sequential
from keras.layers import Dense, Activation
model = Sequential()
model.add(Dense(32, input_dim = 4)) #32個のニューロン、入力4つ Dense:全結合層
model.add(Activation("relu"))
model.add(Dense(32)) #入力総からの受取りはないからinput_dimいらない
model.add(Activation("relu"))
model.add(Dense(3)) #最後の層はクラス数
model.add(Activation("softmax"))
model.compile(optimizer="SGD", loss = "categorical_crossentropy", metrix = ["accuracy"])
print(model.summary())

In [34]:
#Pytorchバージョン
import torch
import torch.nn as nn

In [50]:
#IrisNetクラスを作成、簡単なニューラルネットワーク
class IrisNet(nn.Module):
  def __init__(self):
    super().__init__() #親クラスnn.Moduleの初期化を行っている

    self.model = nn.Sequential(
        nn.Linear(4, 32), #入力4, 出力32(全結合層へ)
        nn.ReLU(), #活性化関数

        nn.Linear(32, 32), #入力32, 出力32
        nn.ReLU(), #活性化関数

        nn.Linear(32, 3) #最後の出力はクラス数
        #最後のsoftmaxは書かず、criterionの中で計算してくれる
    )
  def forward(self, x): #xはそのモデルに必要な入力
    #今回は入力ががく片長、がく片幅、花びら長、花びら幅であり入力は１つだからxだけでいい
      return self.model(x)
model = IrisNet()
print(model)


IrisNet(
  (model): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=3, bias=True)
  )
)


In [42]:
#最適化、損失関数の定義
import torch.optim as optim
criterion = nn.CrossEntropyLoss() #損失関数
optimizer = optim.SGD(model.parameters(), lr = 0.01)

In [45]:
#学習
from torch.utils.data import TensorDataset, DataLoader

#DataSetの作成(datasetではデータを管理している)
train_dataset = TensorDataset(x_train, t_train)
#DataLoaderの作成(datasetから8個まとめる、tensorにして返す。またエポックが変わる、データをシャッフルなどいろいろしてくれうｒ)
train_loader = DataLoader(train_dataset,
                          batch_size = 8,
                          shuffle = True)

In [52]:
#学習
epochs = 30
for epoch in range(epoch): #epochを回す(30回学習)
  running_loss = 0.0 #損失の合計を初期化(1epochの損失を記録)
  for x_batch, t_batch in train_loader: #train_loaderから8こずつ取り出しまくってくれる
    #勾配の初期化
    optimizer.zero_grad()
    #順伝搬
    output = model(x_batch)
    #損失関数
    loss = criterion(output, t_batch)
    #誤差逆伝搬(loss→偏微分→重みがどれくらい悪いか計算してくれる)
    loss.backward()
    #パラメータの更新
    optimizer.step()
    running_loss += loss.item() #print(loss)だとうまくいかないから数値だけとりだせるitemメソッド

    #accuracyの計算





Epoch 29/30, Loss=16.0739


In [ ]:
#学習の推移
import matplotlib.pyplot as plt


まずはデータセットとデータローダの作成  
from torch.utils.data import TensorDataset, DataLoader  

optimizer.zero_grad()      # ① 勾配をリセット  
output = model(x_batch)    # ② 順伝播  
loss = criterion(output, t_batch)  # ③ 損失を計算  
loss.backward()            # ④ 勾配を計算  
optimizer.step()           # ⑤ 重みを更新